### Ingestão de dados na Camada Bronze

Ingestão dos arquivos CSV históricos das tabelas atribuídas disponíveis no container `raw/batch-data` para a camada Bronze.

1. Leitura dos arquivos CSV da camada Raw e captura dos metadados dos arquivos.
2. Verificação da data de modificação do arquivo em relação à última ingestão registrada no log da tabela.
3. Adição das colunas de auditoria `bronze_ingested_at` e `bronze_source_file`.
4. Criação das colunas `ano` e `mes` a partir da data de negócio da tabela.
5. Armazenamento dos dados no formato Delta na camada Bronze, utilizando `append` e particionando por ano e mês.
6. Registro da ingestão no log de metadados, contendo arquivo processado, quantidade de registros, status e timestamp da ingestão.

In [0]:
import os
from dotenv import load_dotenv
from pyspark.sql.functions import col,current_timestamp,year,month,max as spark_max
from pyspark.sql.types import StructType,StructField,StringType,LongType,TimestampType
from pyspark.errors import AnalysisException

In [0]:
# CARREGAMENTO DE VARIÁVEIS DE AMBIENTE
load_dotenv()

client_id = os.getenv("CLIENT_ID")
tenant_id = os.getenv("TENANT_ID")
client_secret = os.getenv("CLIENT_SECRET")
storage_account_name = os.getenv("STORAGE_ACCOUNT_NAME")


# DICIONÁRIO DE CREDENCIAIS OAUTH PARA USAR COM SPARK
adls_options = {
    f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{storage_account_name}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{storage_account_name}.dfs.core.windows.net": client_id,
    f"fs.azure.account.oauth2.client.secret.{storage_account_name}.dfs.core.windows.net": client_secret,
    f"fs.azure.account.oauth2.client.endpoint.{storage_account_name}.dfs.core.windows.net": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}

# DICIONÁRIO DE PATHS
paths = {
    "raw_path": f"abfss://raw@{storage_account_name}.dfs.core.windows.net/batch-data",
    "food_avaliacoes_produto_bronze": f"abfss://squad3@{storage_account_name}.dfs.core.windows.net/bronze/food_avaliacoes_produto",
    "food_estoque_lojas_bronze": f"abfss://squad3@{storage_account_name}.dfs.core.windows.net/bronze/food_estoque_lojas",
    "food_avaliacoes_produto_log": f"abfss://squad3@{storage_account_name}.dfs.core.windows.net/metadata/metadata_ingestion_log/food_avaliacoes_produto",
    "food_estoques_lojas_log": f"abfss://squad3@{storage_account_name}.dfs.core.windows.net/metadata/metadata_ingestion_log/food_estoque_lojas"
}

# DICIONÁRIO DE INFORMAÇÕES DAS TABELAS
tables_config = {
    "food_avaliacoes_produto": {
        "file_pattern": "food_avaliacoes_produto",
        "date_column": "dt_avaliacao",
        "bronze_path": paths["food_avaliacoes_produto_bronze"],
        "log_path": paths["food_avaliacoes_produto_log"]
    },

    "food_estoque_lojas": {
        "file_pattern": "food_estoque_lojas",
        "date_column": "dt_snapshot",
        "bronze_path": paths["food_estoque_lojas_bronze"],
        "log_path": paths["food_estoques_lojas_log"]
    }
}

# SCHEMA DO REGISTRO DE ARQUIVO PROCESSADO
schema_log = StructType([
    StructField("tabela_origem", StringType(), False),
    StructField("arquivo_processado", StringType(), False),
    StructField("total_linhas", LongType(), False),
    StructField("status", StringType(), False),
    StructField("bronze_ingested_at", TimestampType(), False)
])

In [0]:
# INGESTÃO BRONZE COM CONTROLE POR TIMESTAMP
def ingest_bronze(table_name: str):
    # 1. Capturar metadata do arquivo fonte
    file_info = (
        spark.read
        .format("binaryFile")
        .options(**adls_options)
        .load(paths["raw_path"])
        .filter(col("path").contains(tables_config[table_name]["file_pattern"]))
        .select(
            col("path").alias("file_path"),
            col("modificationTime").alias("modified_at")
        )
    )

    file_mod_time = file_info.select("modified_at").first()["modified_at"]
    processed_file = file_info.select("file_path").first()["file_path"]

    # 2. Verificar última ingestão registrada no log
    try:
        df_log = (
            spark.read
            .format("delta")
            .options(**adls_options)
            .load(tables_config[table_name]["log_path"])
        )

        last_ingestion = df_log.agg(
            spark_max("bronze_ingested_at").alias("last")
        ).first()["last"]

        if last_ingestion is not None and file_mod_time <= last_ingestion:
            print(f"[SEM ALTERAÇÃO] {table_name} não possui novos dados para ingestão.")
            return

    except Exception:
        last_ingestion = None

    # 3. Ler CSV
    df = (
        spark.read
        .options(**adls_options)
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(processed_file)
    )

    raw_count = df.count()

    # 4. Adicionar colunas de auditoria
    df = (
        df
        .withColumn("bronze_ingested_at", current_timestamp())
        .withColumn("bronze_source_file", col("_metadata.file_path"))
    )

    # 5. Criar colunas de partição
    df = (
        df
        .withColumn("ano", year(col(tables_config[table_name]["date_column"])))
        .withColumn("mes", month(col(tables_config[table_name]["date_column"])))
    )

    # 6. Salvar Delta append
    (
        df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .partitionBy("ano", "mes")
        .options(**adls_options)
        .save(tables_config[table_name]["bronze_path"])
    )

    # 7. Capturar timestamp da ingestão
    bronze_ingested_at = df.agg(
        spark_max("bronze_ingested_at").alias("bronze_ingested_at")
    ).first()["bronze_ingested_at"]

    # 8. Criar registro do log
    df_log_new = spark.createDataFrame(
        [(table_name, processed_file, raw_count, "SUCCESS", bronze_ingested_at)],
        schema=schema_log
    )

    # 9. Salvar log da tabela
    (
        df_log_new.write
        .format("delta")
        .mode("append")
        .options(**adls_options)
        .save(tables_config[table_name]["log_path"])
    )

    print(f"[INGESTÃO CONCLUÍDA] {table_name} | {raw_count} registros")

In [0]:
# EXECUÇÃO DAS TABELAS BRONZE

df_bronze_avaliacoes = ingest_bronze("food_avaliacoes_produto")
df_bronze_estoque = ingest_bronze("food_estoque_lojas")

# VISUALIZAÇÃO DOS DATAFRAMES BRONZE
display(df_bronze_avaliacoes)
display(df_bronze_estoque)